# MSE Gaussian LOO

This notebook loads the tuning artifact produced by `preprocessing_LOO_gaussian.ipynb` and estimates the MSE of the LOO estimators without recomputing tuning parameters.

The x-axis budget is the final estimation budget `N_i * M_lambda`; preprocessing/tuning cost is excluded.

In [ ]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gaussian_LOO import BayesianLinearRegressionTempering
from src.normal import uestimator_given_lambda, q_unbiased_estimator
from src.startified_estimator import optimal_startified_estimator


## Load Precomputed Tuning

In [ ]:
tuning_path = Path("results/gaussian_loo_tuning_Mlambda100.pkl")

with tuning_path.open("rb") as f:
    artifact = pickle.load(f)

config = artifact["config"]
data = artifact["data"]
tuning_by_index = artifact["tuning_by_index"]

X_data = data["X_data"]
y_data = data["y_data"]
prior_mean = data["prior_mean"]
prior_cov = data["prior_cov"]
sigma2_noise = data["sigma2_noise"]
true_value = data["true_value"]

n = config["n"]
L = config["L"]
lambda_grid = config["lambda_grid"]
M_lambda = config["M_lambda"]
sigmaq = config["sigmaq"]
lag = config["lag"]

blr_path = BayesianLinearRegressionTempering(
    X_data,
    y_data,
    prior_mean,
    prior_cov,
    sigma2_noise,
)

print(f"Loaded tuning for {len(tuning_by_index)} indices")
print(f"Fixed M_lambda: {M_lambda}")
print(f"Exact conjugate LOO ELPD: {true_value:.6f}")


## MSE Configuration

`nrep_mse` controls the Monte Carlo precision of the estimated MSE curve. It is not part of the budget of one estimator.

In [ ]:
SEED_MSE = 2028
rng_mse = np.random.default_rng(SEED_MSE)
np.random.seed(SEED_MSE)

N_i_grid = np.array([5, 10, 20, 40, 80])
nrep_mse = 20
sample_with_replacement = True


## Helper Functions

In [ ]:
def make_loo_estimator(index_i):
    tuned = tuning_by_index[int(index_i)]
    log_target_path_i = lambda theta, path, i=int(index_i): blr_path.log_path(
        theta,
        path,
        i,
    )
    grad_log_target_path_i = lambda theta, i=int(index_i): blr_path.log_likelihood_i(theta, i)

    return lambda lam: uestimator_given_lambda(
        lam,
        lambda_grid,
        tuned["k_grid"],
        tuned["m_grid"],
        sigmaq,
        lag,
        log_target_path_i,
        grad_log_target_path_i,
    )


def sample_loo_indices(N_i):
    if sample_with_replacement:
        return rng_mse.integers(0, n, size=N_i)
    if N_i > n:
        raise ValueError("N_i cannot exceed n when sampling without replacement.")
    return rng_mse.choice(n, size=N_i, replace=False)


## Run MSE Experiment

In [ ]:
rows = []
start_time = time.perf_counter()

for N_i in N_i_grid:
    print(f"Running N_i={N_i}, final budget={N_i * M_lambda}")
    estimates_is = np.zeros(nrep_mse)
    estimates_strat = np.zeros(nrep_mse)

    for r in range(nrep_mse):
        sampled_indices = sample_loo_indices(int(N_i))
        loo_integrals_is = np.zeros(int(N_i))
        loo_integrals_strat = np.zeros(int(N_i))

        for i_draw, index_i in enumerate(sampled_indices):
            tuned = tuning_by_index[int(index_i)]
            estimator_i = make_loo_estimator(index_i)

            loo_integrals_is[i_draw] = q_unbiased_estimator(
                L,
                M_lambda,
                estimator_i,
                tuned["sqrt_m2"],
            )
            loo_integrals_strat[i_draw] = optimal_startified_estimator(
                L,
                estimator_i,
                tuned["budget"],
            )

        estimates_is[r] = n * np.mean(loo_integrals_is)
        estimates_strat[r] = n * np.mean(loo_integrals_strat)

    for label, estimates in [
        ("Importance sampling", estimates_is),
        ("Stratified sampling", estimates_strat),
    ]:
        rows.append({
            "N_i": int(N_i),
            "M_lambda": int(M_lambda),
            "final_budget": int(N_i * M_lambda),
            "estimator": label,
            "mean": np.mean(estimates),
            "bias": np.mean(estimates) - true_value,
            "variance": np.var(estimates, ddof=1),
            "mse": np.mean((estimates - true_value) ** 2),
        })

elapsed_mse = time.perf_counter() - start_time
df_mse = pd.DataFrame(rows)
print(f"MSE experiment time, tuning excluded: {elapsed_mse:.2f} seconds")
df_mse


## Plot MSE

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)

for label, group in df_mse.groupby("estimator"):
    ax.plot(
        group["final_budget"],
        group["mse"],
        marker="o",
        linewidth=2,
        label=label,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Final budget $N_i M_\lambda$")
ax.set_ylabel("MSE")
ax.set_title("Gaussian LOO MSE, tuning excluded")
ax.grid(True, which="both", alpha=0.4)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Save MSE Results

In [ ]:
results_path = Path("results") / f"gaussian_loo_mse_Mlambda{M_lambda}.csv"
df_mse.to_csv(results_path, index=False)
print(f"Saved MSE results to: {results_path}")
